# Baseline Centralized Model Training

This notebook trains a centralized neural network model on the CICIDS2017 dataset as a baseline.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

from model import NeuralNetwork
from dataset import load_processed_data, DataLoader
from evaluate import ModelEvaluator

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries and modules imported successfully")

## 1. Load Preprocessed Data

In [ ]:
# Load data
data_dir = "../data/processed"
X_train, X_test, y_train, y_test = load_processed_data(data_dir)

print(f"Training data: {X_train.shape}")
print(f"Test data: {X_test.shape}")
print(f"\nTrain labels - Positive: {np.sum(y_train == 1)}, Negative: {np.sum(y_train == 0)}")
print(f"Test labels - Positive: {np.sum(y_test == 1)}, Negative: {np.sum(y_test == 0)}")

## 2. Create and Train Model

In [ ]:
# Hyperparameters
input_dim = X_train.shape[1]
hidden_dims = [128, 64]
output_dim = 1
learning_rate = 0.01
batch_size = 32
epochs = 50

print(f"Model Configuration:")
print(f"  Input dim: {input_dim}")
print(f"  Hidden dims: {hidden_dims}")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size: {batch_size}")
print(f"  Epochs: {epochs}")

# Create model
model = NeuralNetwork(input_dim=input_dim, hidden_dims=hidden_dims, 
                      output_dim=output_dim, learning_rate=learning_rate)
print(f"\nModel size: {model.get_model_size()} parameters")

In [ ]:
# Training
train_losses = []

for epoch in range(epochs):
    loader = DataLoader(X_train, y_train, batch_size=batch_size, shuffle=True)
    
    epoch_loss = 0
    num_batches = 0
    
    for X_batch, y_batch in loader:
        y_batch = y_batch.reshape(-1, 1)
        loss = model.train_step(X_batch, y_batch)
        epoch_loss += loss
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    train_losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{epochs}: Loss = {avg_loss:.4f}")

print(f"\nTraining completed!")

## 3. Training Loss Visualization

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, epochs + 1), train_losses, marker='o', linewidth=2, markersize=4)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss Over Epochs', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nLoss Statistics:")
print(f"  Initial loss: {train_losses[0]:.4f}")
print(f"  Final loss: {train_losses[-1]:.4f}")
print(f"  Improvement: {(train_losses[0] - train_losses[-1]) / train_losses[0] * 100:.2f}%")

## 4. Model Evaluation

In [ ]:
# Evaluate
evaluator = ModelEvaluator()
metrics = evaluator.evaluate_model(model, X_test, y_test)
evaluator.print_metrics(metrics)

## 5. Metrics Visualization

In [ ]:
# Plot metrics
metric_names = ['accuracy', 'precision', 'recall', 'f1', 'specificity']
metric_values = [metrics[m] for m in metric_names]

colors = ['#3498db', '#e74c3c', '#f39c12', '#27ae60', '#9b59b6']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = ax1.bar(metric_names, metric_values, color=colors, alpha=0.7, edgecolor='black')
for bar, value in zip(bars, metric_values):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_ylabel('Score', fontsize=11)
ax1.set_title('Performance Metrics', fontsize=12, fontweight='bold')
ax1.set_ylim(0, 1.1)
ax1.grid(axis='y', alpha=0.3)

# Confusion matrix heatmap
cm = metrics.get('confusion_matrix', {})
cm_array = np.array([
    [cm.get('TP', 0), cm.get('FP', 0)],
    [cm.get('FN', 0), cm.get('TN', 0)]
])

sns.heatmap(cm_array, annot=True, fmt='d', cmap='Blues', ax=ax2, cbar=True,
           xticklabels=['Positive', 'Negative'],
           yticklabels=['Positive', 'Negative'])
ax2.set_xlabel('Predicted', fontsize=11)
ax2.set_ylabel('Actual', fontsize=11)
ax2.set_title('Confusion Matrix', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Summary

In [ ]:
print("\n" + "="*60)
print("BASELINE MODEL SUMMARY")
print("="*60)
print(f"Architecture: {input_dim} -> {hidden_dims} -> {output_dim}")
print(f"Parameters: {model.get_model_size():,}")
print(f"Training epochs: {epochs}")
print(f"Batch size: {batch_size}")
print(f"\nFinal Metrics:")
print(f"  Accuracy:  {metrics['accuracy']:.4f}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall:    {metrics['recall']:.4f}")
print(f"  F1-Score:  {metrics['f1']:.4f}")
print(f"\nConfusion Matrix:")
print(f"  TP: {cm.get('TP', 0)}, FP: {cm.get('FP', 0)}")
print(f"  FN: {cm.get('FN', 0)}, TN: {cm.get('TN', 0)}")
print("="*60)